# OEC MusicGen — Gradio music worker

Kernel: phase1. Backend: AudioCraft 1.3.0 / Torch 2.1.0 + CUDA 11.8. The UI invokes the existing CLI in offline/cache-only mode and writes a bounded test WAV under /tmp. MusicGen weights are CC-BY-NC 4.0; do not route outputs to commercial use without rights review.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile
import uuid
import gradio as gr

ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline_musicgen.py").is_file():
    ROOT = Path(os.environ.get("OEC_REPO_ROOT", str(ROOT))).expanduser().resolve()
if not (ROOT / "pipeline_musicgen.py").is_file():
    raise RuntimeError("Run in the OEC repo root or set OEC_REPO_ROOT.")

def generate_music(prompt, duration, seed):
    if not isinstance(prompt, str) or not prompt.strip():
        raise gr.Error("Enter a music prompt.")
    if not 1 <= int(duration) <= 10:
        raise gr.Error("Duration must be 1–10 seconds for this bounded smoke UI.")
    output = Path(tempfile.gettempdir()) / f"oec-musicgen-{uuid.uuid4().hex}.wav"
    cmd = [sys.executable, str(ROOT / "pipeline_musicgen.py"), "--mode", "generate",
           "--offline", "--scratch-output", "--output", str(output),
           "--prompt", prompt.strip(), "--duration", str(int(duration)), "--seed", str(int(seed))]
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, timeout=1200, check=False)
    log = (result.stdout + "\n" + result.stderr)[-8000:]
    if result.returncode:
        raise gr.Error("MusicGen failed; offline mode prevented model downloads.\n" + log)
    if not output.is_file() or output.stat().st_size == 0:
        raise gr.Error("MusicGen returned no non-empty WAV.")
    info = output.with_suffix(".info.json")
    return str(output), info.read_text(encoding="utf-8") if info.is_file() else log

demo = gr.Interface(fn=generate_music,
    inputs=[gr.Textbox(label="Music prompt", lines=3, value="Calm instrumental soundtrack with warm piano and soft strings"),
            gr.Slider(1, 10, value=3, step=1, label="Duration (seconds)"), gr.Number(value=42, precision=0, label="Seed")],
    outputs=[gr.Audio(type="filepath", label="Generated WAV"), gr.Textbox(label="Run metadata/logs", lines=10)],
    title="OEC MusicGen", description="Offline/cache-only AudioCraft music generation; research/non-commercial weight licence.")
demo.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7902, share=False, inline=False)
